# 2026-06-19 Unified phage RBP reference database

Aggregates **all receptor-binding proteins (RBPs)** described by three literature
sources into one machine-readable database (FASTA + metadata TSV). The RBP
*sequences* are **not** in the PDF bodies — each paper deposits them externally,
so this notebook fetches/normalizes those deposits.

| Source | Deposit (`raw/`) | What it gives |
|---|---|---|
| **Boeckaerts 2021**, *Sci Rep* | GitHub `dimiboeckaerts/BacteriophageHostPrediction` → `RBP_database.csv` | collected RBPs across 9 hosts, with host + accession |
| **Krusche**, *Cell Host & Microbe* (PhARIS) | GitHub `JKrusche1/PhARIS` → `files/RBPs_main`, `files/RBPs_rev` | *S. aureus* phage RBPs, clustered (main = irreversible, rev = reversible); all bind WTA |
| **Moriniere 2026**, bioRxiv (GenoPHI) | Figshare `10.6084/m9.figshare.31930314` → Datasets S1–S7 / Tables S1–S10 | *E. coli* phage RBPs by receptor class |

**Method.** Boeckaerts and Krusche are direct parses. For Moriniere the
supplementary data is *feature-centric* (Dataset S6 lists every protein carrying a
receptor-predictive k-mer — ~20 proteins/phage, not all RBPs), so the RBP per
`(phage × receptor)` is taken as the protein carrying the **most predictive k-mers**
for that receptor (RBPs concentrate specificity determinants), after assigning each
protein to its single argmax-feature receptor (prevents one protein being smeared
across receptors). Receptor assignments are then annotated with curated provenance:
`experimentally_validated` (Tables S8/S9), `af3_validated` (Table S5, OMP-only),
else `kmer_candidate`. Dedup uses **MMseqs2** `easy-cluster` (same tool as
`code/output/0617_receptor_similarity/`).

**Reproducing the downloads** (already cached in `raw/`):
```bash
curl -sL -o raw/RBP_database.csv https://raw.githubusercontent.com/dimiboeckaerts/BacteriophageHostPrediction/master/RBP_database.csv
git clone --depth 1 https://github.com/JKrusche1/PhARIS.git raw/PhARIS
curl -sL -o raw/EcoliPhage_Datasets_S1-S7.xlsx https://ndownloader.figshare.com/files/63834690
curl -sL -o raw/EcoliPhage_Tables_S1-S10.xlsx  https://ndownloader.figshare.com/files/63834696
```

In [1]:
import json, re, shutil, subprocess
from pathlib import Path
import pandas as pd

RAW = Path("raw")                                   # cwd = this notebook's dir
OUT = Path("../output/0619_literature_rbp_database")
OUT.mkdir(parents=True, exist_ok=True)
MMSEQS = shutil.which("mmseqs"); assert MMSEQS, "mmseqs not on PATH"

COLS = ["rbp_id","source","phage_name","host_species","rbp_type","cluster",
        "receptor","receptor_class","accession","seq_length","is_pseudomonas",
        "confidence","protein_sequence"]

def parse_fasta(path):
    out, hdr, buf = [], None, []
    for line in open(path):
        line = line.rstrip("\n")
        if line.startswith(">"):
            if hdr is not None: out.append((hdr, "".join(buf)))
            hdr, buf = line[1:], []
        else: buf.append(line.strip())
    if hdr is not None: out.append((hdr, "".join(buf)))
    return out

def rbp_type_from_name(name):
    n = str(name).lower()
    if "spike" in n: return "tail spike"
    if "baseplate" in n: return "baseplate"
    if "fiber" in n or "fibre" in n: return "tail fiber"
    if "uncharacter" in n or n in ("nan","none",""): return None
    return "tail fiber/other"

def species_only(host):
    t = str(host).split(); return " ".join(t[:2]) if len(t) >= 2 else str(host)

## 1 · Boeckaerts 2021 — collected RBPs, 9 hosts

In [2]:
df = pd.read_csv(RAW/"RBP_database.csv")
rows = []
for i, r in df.iterrows():
    seq = str(r["protein_sequence"]).strip().replace("*","")
    rows.append(dict(
        rbp_id=f"boeckaerts_{i:04d}", source="boeckaerts2021",
        phage_name=r.get("org"), host_species=species_only(r["host"]),
        rbp_type=rbp_type_from_name(r.get("name")), cluster=None,
        receptor=None, receptor_class=None,
        accession=r.get("embl_id") if pd.notna(r.get("embl_id")) else r.get("id"),
        seq_length=len(seq), is_pseudomonas="pseudomonas" in str(r["host"]).lower(),
        confidence="literature_annotated", protein_sequence=seq))
boeck = pd.DataFrame(rows)
print(len(boeck), "RBPs |", int(boeck.is_pseudomonas.sum()), "Pseudomonas")
boeck.host_species.value_counts().head(10)

1232 RBPs | 156 Pseudomonas


host_species
Escherichia coli           399
Klebsiella pneumoniae      227
Pseudomonas aeruginosa     156
Salmonella enterica        148
Staphylococcus aureus       95
Acinetobacter baumannii     69
Clostridium difficile       42
Enterobacter cloacae        29
Shigella sonnei             12
Shigella sp.                 9
Name: count, dtype: int64

## 2 · Krusche / PhARIS — *S. aureus* phage RBPs (clustered)

In [3]:
rows, n = [], 0
for sub, rtype in [("RBPs_main","main (irreversible)"), ("RBPs_rev","reversible")]:
    for fa in sorted((RAW/"PhARIS"/"files"/sub).glob("*.fasta")):
        cluster = fa.stem.replace("_reversible","")
        for hdr, seq in parse_fasta(fa):
            pid = hdr.split()[0]; seq = seq.strip().replace("*","")
            if not seq: continue
            rows.append(dict(
                rbp_id=f"krusche_{n:04d}", source="krusche_chm",
                phage_name=re.sub(r"_orf\d+.*$","",pid),
                host_species="Staphylococcus aureus", rbp_type=rtype, cluster=cluster,
                receptor="WTA", receptor_class="wall teichoic acid (polysaccharide)",
                accession=pid, seq_length=len(seq), is_pseudomonas=False,
                confidence="literature_annotated", protein_sequence=seq)); n += 1
krusche = pd.DataFrame(rows)
print(len(krusche), "RBPs |", krusche.phage_name.nunique(), "phages |",
      krusche.cluster.nunique(), "main-clusters")

575 RBPs | 339 phages | 8 main-clusters


## 3 · Moriniere / GenoPHI — *E. coli* phage RBPs

Dataset S6 is feature-centric; reduce to the top-k-mer protein per
`(phage × receptor)` (argmax-receptor first), then annotate curated provenance.

In [4]:
RECEPTOR_CLASS = {
    "ompa":"OMP","ompc":"OMP","ompf":"OMP","ompw":"OMP","btub":"OMP","fhua":"OMP",
    "lamb":"OMP","tolc":"OMP","tsx":"OMP","fadl":"OMP","lptd":"OMP","yncd":"OMP","nupg":"OMP",
    "kdo":"LPS core sugar","hepi":"LPS core sugar","hepii":"LPS core sugar","glui":"LPS core sugar",
    "ngr":"N4-glycan receptor polysaccharide"}
def norm(x): return re.sub(r"[*\s]","",str(x)).lower()

xls_d, xls_t = RAW/"EcoliPhage_Datasets_S1-S7.xlsx", RAW/"EcoliPhage_Tables_S1-S10.xlsx"
s6 = pd.read_excel(xls_d, sheet_name="Dataset S6", header=1)
seqmap = s6.drop_duplicates("Protein_ID").set_index("Protein_ID")["Full_sequence"]
fc = s6.groupby(["Protein_ID","Phage","Receptor"]).size().reset_index(name="nfeat")
fc["len"] = fc["Protein_ID"].map(seqmap.str.len())
prot = fc.sort_values(["nfeat","len"], ascending=False).drop_duplicates("Protein_ID")     # argmax receptor
rbp  = prot.sort_values(["nfeat","len"], ascending=False).drop_duplicates(["Phage","Receptor"])  # RBP per pair

# curated provenance
af3 = pd.read_excel(xls_t, sheet_name="Table S5", header=1)
af3_pairs = {(norm(p),norm(r)) for p,r in zip(af3.Phage, af3.Receptor)}
exp_pairs = set()
for sh in ["Table S8","Table S9"]:
    t = pd.read_excel(xls_t, sheet_name=sh, header=[1,2])
    t.columns = [" / ".join(str(x) for x in c if "Unnamed" not in str(x)).strip() for c in t.columns]
    for col in ["Experimental validation / Primary receptor","Experimental validation / LPS core sugar receptor"]:
        for p,r in zip(t.Phage, t[col]):
            if pd.notna(r) and norm(r) not in ("","-"): exp_pairs.add((norm(p),norm(r)))

rows = []
for i, r in rbp.reset_index(drop=True).iterrows():
    key = (norm(r.Phage), norm(r.Receptor))
    conf = ("experimentally_validated" if key in exp_pairs else
            "af3_validated" if key in af3_pairs else "kmer_candidate")
    seq = str(seqmap[r.Protein_ID]).strip().replace("*","")
    rows.append(dict(
        rbp_id=f"moriniere_{i:04d}", source="moriniere2026", phage_name=r.Phage,
        host_species="Escherichia coli", rbp_type="RBP (top k-mer-feature protein)",
        cluster=None, receptor=r.Receptor, receptor_class=RECEPTOR_CLASS.get(norm(r.Receptor), r.Receptor),
        accession=r.Protein_ID, seq_length=len(seq), is_pseudomonas=False,
        confidence=conf, protein_sequence=seq))
mori = pd.DataFrame(rows)
print(len(mori), "RBPs |", mori.phage_name.nunique(), "phages")
mori.confidence.value_counts()

640 RBPs | 241 phages


confidence
kmer_candidate              467
af3_validated               107
experimentally_validated     66
Name: count, dtype: int64

## 4 · Assemble + dedup (MMseqs2) + write outputs

In [5]:
both = pd.concat([boeck, krusche, mori], ignore_index=True)[COLS]
bad = both.protein_sequence.str.len().fillna(0) < 10
both = both[~bad].reset_index(drop=True)
both["seq_length"] = both.protein_sequence.str.len()

def write_fasta(df, path):
    with open(path,"w") as fh:
        for _,r in df.iterrows(): fh.write(f">{r.rbp_id}\n{r.protein_sequence}\n")

def cluster(fasta, tag, min_id, cov=0.8):
    pref, tmp = OUT/f"_clu_{tag}", OUT/f"_tmp_{tag}"
    subprocess.run([MMSEQS,"easy-cluster",str(fasta),str(pref),str(tmp),
                    "--min-seq-id",str(min_id),"-c",str(cov),"--cov-mode","0","-v","1"],
                   check=True, capture_output=True)
    m = {mem:rep for rep,mem in (l.rstrip("\n").split("\t") for l in open(f"{pref}_cluster.tsv"))}
    shutil.rmtree(tmp, ignore_errors=True)
    for p in OUT.glob(f"_clu_{tag}_*seq*.fasta"): p.unlink()
    return m

all_fa = OUT/"_all.fasta"; write_fasta(both, all_fa)
both["cluster_rep_100"] = both.rbp_id.map(cluster(all_fa,"id100",1.0))
both["cluster_rep_95"]  = both.rbp_id.map(cluster(all_fa,"id95",0.95))
uniq = both[both.rbp_id == both.cluster_rep_100].reset_index(drop=True)

write_fasta(uniq, OUT/"rbp_database.fasta")
both.to_csv(OUT/"rbp_metadata.tsv", sep="\t", index=False)
both[["rbp_id","source","host_species","cluster_rep_100","cluster_rep_95"]].to_csv(
    OUT/"rbp_clusters.tsv", sep="\t", index=False)

# Pseudomonas slice for the binding screen (code/0616_binding-screen-fold-plan.md)
pse = uniq[uniq.is_pseudomonas]
write_fasta(pse, OUT/"pseudomonas_rbp_candidates.fasta")

all_fa.unlink()
for p in OUT.glob("_clu_*"): p.unlink()
print(f"{len(both)} rows -> {len(uniq)} unique seqs ({len(both)-len(uniq)} exact dups removed)")
print(f"Pseudomonas slice: {len(pse)} seqs -> pseudomonas_rbp_candidates.fasta")

2447 rows -> 1673 unique seqs (774 exact dups removed)
Pseudomonas slice: 115 seqs -> pseudomonas_rbp_candidates.fasta


## 5 · Summary & verification

In [6]:
print("=== unique DB by source ===");      print(uniq.source.value_counts())
print("\n=== unique DB by host (top 10) ==="); print(uniq.host_species.value_counts().head(10))
print("\n=== Moriniere receptor-class x confidence ===")
print(pd.crosstab(mori.receptor_class, mori.confidence))

# cross-source 95% redundancy
g = both.groupby("cluster_rep_95").source.nunique(); mixed = g[g>1].index
xs = (both[both.cluster_rep_95.isin(mixed)].groupby("cluster_rep_95").source
      .agg(lambda s: tuple(sorted(set(s)))).value_counts())
print("\n=== cross-source 95%-identity overlap (clusters) ==="); print(xs)

# integrity assertions
assert both.host_species.notna().all()
assert (both.protein_sequence.str.len() >= 10).all()
assert (both.seq_length == both.protein_sequence.str.len()).all()
assert sum(1 for l in open(OUT/"rbp_database.fasta") if l.startswith(">")) == len(uniq)
print("\nintegrity OK — fasta seqs ==", len(uniq))

# spot-checks vs papers
print("\nK1F tailspike (Boeckaerts):",
      uniq[uniq.accession.astype(str).str.contains("Q04830|AJ505988",na=False)][["phage_name","rbp_type","seq_length"]].to_dict("records"))
print("EV240 gp38->OmpA (Moriniere, expt):",
      both[(both.accession=="CQHXHGTY_CDS_0087")][["receptor","confidence","seq_length"]].to_dict("records"))

=== unique DB by source ===
source
boeckaerts2021    933
moriniere2026     474
krusche_chm       266
Name: count, dtype: int64

=== unique DB by host (top 10) ===
host_species
Escherichia coli           792
Staphylococcus aureus      320
Klebsiella pneumoniae      175
Pseudomonas aeruginosa     115
Salmonella enterica        114
Acinetobacter baumannii     64
Clostridium difficile       21
Enterobacter cloacae        17
Shigella sonnei             10
Shigella sp.                 7
Name: count, dtype: int64

=== Moriniere receptor-class x confidence ===
confidence                         af3_validated  experimentally_validated  \
receptor_class                                                               
LPS core sugar                                 0                        18   
N4-glycan receptor polysaccharide              0                         5   
OMP                                          107                        43   

confidence                         kmer_candidate 